# VSSD + First Impressions v2

Цель:
- использовать архитектуру **VSSD (Vision Mamba with Non-Causal State Space Duality)** как визуальный бэкбон;
- обучить видео-модель для регрессии по пяти персональным качествам (Big Five) на датасете **First Impressions v2**;
- подготовить базовые визуализации (распределение таргетов, кривые обучения, scatter-плоты предсказаний).

Датасет:
- структура: DATA/TRAIN, DATA/VALIDATION, DATA/TEST
- внутри каждой папки:
    - mp4-ролики
    - папка Annotation/ с двумя .pkl файлами:
        - аннотации (Big Five + interview) – dict of dicts
        - транскрипции – dict (video_name → текст)

Модель:
- VSSD используется как бэкбон для извлечения признаков из кадров (из статьи: некаузальный NC-SSD, глобальное состояние, инвариантность к направлению сканирования, сохранение 2D-структуры изображения).
- поверх бэкбона – простая временная агрегация и регрессионная голова на 5 признаков.


**Важно (текущая версия):**
- VSSD-бэкбон используется только для одноразовой выгрузки визуальных признаков из видео в папку `features_vssd/`.
- Основное обучение модели идёт по сохранённым признакам (по `.pt` файлам), что сильно ускоряет обучение и уменьшает нагрузку на диск.

## 1. Базовая конфигурация и окружение

Здесь:
- задаём пути к данным,
- проверяем доступные GPU,
- включаем смешанную точность (fp16/bf16),
- закладываем возможность запуска в DDP (torchrun), но код будет работать и на одной GPU.


In [1]:
import sys
! rm -rf ~/.local/lib/python3.9/site-packages/torch*
! rm -rf ~/.local/lib/python3.9/site-packages/nvidia*
! rm -rf ~/.local/lib/python3.9/site-packages/triton*
# ОДИН РАЗ: фикс конфликтов с бинарниками под NumPy 1.x
!{sys.executable} -m pip install --user --no-cache-dir "numpy<2" --force-reinstall


Looking in indexes: http://nexus.charisma:5080/repository/pypi-proxy/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 309.3 MB/s  0:00:00
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.56.4 requires numpy<1.24,>=1.18, but you have numpy 1.26.4 which is incompatible.


In [2]:
from pathlib import Path
import sys, site

print("Python:", sys.executable)

# --- 1. user-site в начало sys.path ---
user_site = site.getusersitepackages()
print("USER_SITE:", user_site)

if user_site in sys.path:
    sys.path.remove(user_site)
sys.path.insert(0, user_site)

print("sys.path[0]:", sys.path[0])

Python: /opt/software/python/envs/olymp_2025/bin/python
USER_SITE: /home/eggrankina/.local/lib/python3.9/site-packages
sys.path[0]: /home/eggrankina/.local/lib/python3.9/site-packages


In [3]:
import sys

# 1. Базовые пакеты (устанавливаются в ~/.local один раз)
!{sys.executable} -m pip install --user --no-cache-dir \
    "torch==2.1.0" torchvision torchaudio \
    "timm==0.4.12" pytest chardet termcolor submitit \
    tensorboardX fvcore seaborn opencv-python tensorboard einops

# 2. mamba-ssm ставим ОТДЕЛЬНО
!{sys.executable} -m pip install --user --no-cache-dir "mamba-ssm==2.2.5"

# Чистим кеш pip, чтобы не забивать квоту
!rm -rf ~/.cache/pip


Looking in indexes: http://nexus.charisma:5080/repository/pypi-proxy/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 670.2/670.2 MB 261.1 MB/s  0:00:02a 0:00:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 150.0 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 98.0 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 218.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 66.3 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 206.2 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 253.7 MB/s  0:00:0300:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 257.0 MB/s  0:00:01a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 179.3 MB/s  0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 239.8 MB/s  0:00:00eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import torch

print("torch version:", torch.__version__)
print("torch from:", torch.__file__)

torch version: 2.1.0+cu121
torch from: /home/eggrankina/.local/lib/python3.9/site-packages/torch/__init__.py


In [5]:
from pathlib import Path
import sys

import torch
from torch.utils import _pytree as torch_pytree

# --- Патч для transformers: добавляем register_pytree_node, если его нет ---
if not hasattr(torch_pytree, "register_pytree_node"):
    # transformers использует эту функцию при импорте, нам достаточно заглушки
    def register_pytree_node(cls, flatten_fn, unflatten_fn, *, serialized_type_name=None):
        return
    torch_pytree.register_pytree_node = register_pytree_node

# --- VSSD: путь к classification ---
VSSD_CLASSIFICATION_ROOT = Path("../VSSD/classification")
assert VSSD_CLASSIFICATION_ROOT.exists(), "Укажи правильный путь к VSSD/classification"

if str(VSSD_CLASSIFICATION_ROOT) not in sys.path:
    sys.path.insert(0, str(VSSD_CLASSIFICATION_ROOT))

from config import _C, get_config
from models import build_model

# --- Triton-опы из mamba_ssm, прокидываем в mamba2 ---
from mamba_ssm.ops.triton.ssd_combined import (
    mamba_chunk_scan_combined,
    mamba_split_conv1d_scan_combined,
)
from mamba_ssm.ops.triton.layernorm_gated import RMSNorm as RMSNormGated
from mamba_ssm.ops.triton.selective_state_update import selective_state_update

import models.mamba2 as mamba2

mamba2.mamba_chunk_scan_combined = mamba_chunk_scan_combined
mamba2.mamba_split_conv1d_scan_combined = mamba_split_conv1d_scan_combined
mamba2.RMSNormGated = RMSNormGated
mamba2.selective_state_update = selective_state_update

print("Импорты VSSD + mamba_ssm успешно прошли")


/opt/software/python/envs/olymp_2025/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


mamba_ssm not found
Импорты VSSD + mamba_ssm успешно прошли


In [ ]:
# %%
import os
import math
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import random
import numpy as np
import torch
import torch.nn as nn
from torch.cuda import amp
from torch.cuda.amp import autocast, GradScaler   # ДОБАВЬ ЯВНО
torch.backends.cudnn.benchmark = True

from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler

from torchvision import transforms

import matplotlib.pyplot as plt

# Для воспроизводимости
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# Пути к данным (подстрой под свою структуру)
DATA_ROOT = Path("../DATA")
TRAIN_DIR = DATA_ROOT / "TRAIN"
VAL_DIR = DATA_ROOT / "VALIDATION"
TEST_DIR = DATA_ROOT / "TEST"

assert TRAIN_DIR.exists(), f"Папка {TRAIN_DIR} не найдена"
assert VAL_DIR.exists(), f"Папка {VAL_DIR} не найдена"

# Проверим GPU
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Current device:", torch.cuda.current_device())
    print("Device name:", torch.cuda.get_device_name())


### Рекомендованная вычислительная конфигурация (для суперкомпьютера)

*Это текстовая рекомендация, не код (подстрой под свои кластеры):*

- 4–8× GPU уровня A100 / H100 / V100 32GB.
- Формат запуска: `torchrun --nproc_per_node=N ...`.
- Параметры:
  - разрешение кадров: 224×224 (как в ImageNet-конфигурациях VSSD);
  - число кадров на видео: 16–32;
  - batch size на GPU: 4–8 роликов (в зависимости от памяти);
  - mixed precision (`torch.cuda.amp.autocast`) + `GradScaler`.
- Для первых экспериментов можно начать с:
  - 1 GPU, 16 кадров, batch size = 2–4, чтобы отладить пайплайн.


## 2. Исследование файлов Annotation (.pkl)

В каждой фазе (train/val/test) внутри `Annotation/` лежат два файла:
- аннотации (Big Five + interview) – словарь словарей:
  - `annotation['extraversion'][video_name] -> float in [0, 1]`
  - `annotation['agreeableness'][video_name] -> float`
  - ...
  - `annotation['interview'][video_name] -> float`
- транскрипции – словарь:
  - `transcription[video_name] -> unicode строка`

Ниже – код, который:
- находит .pkl-файлы,
- загружает их,
- печатает примеры.


In [42]:
import pickle

def load_annotation_and_transcription(phase_dir: Path):
    annot_dir = phase_dir / "Annotation"
    assert annot_dir.exists(), f"Нет папки Annotation в {phase_dir}"

    pkl_files = list(annot_dir.glob("*.pkl"))
    assert len(pkl_files) == 2, f"Ожидалось 2 pkl-файла, найдено {len(pkl_files)} в {annot_dir}"

    # Определим, какой из файлов – аннотации, а какой – транскрипции
    def is_annotation_dict(obj):
        # простой эвристический тест структуры
        if not isinstance(obj, dict):
            return False
        # ключи верхнего уровня должны быть строками с именами признаков
        sample_keys = list(obj.keys())
        if not sample_keys:
            return False
        # значения – тоже словари
        return isinstance(obj[sample_keys[0]], dict)

    annotation = None
    transcription = None

    for pkl_path in pkl_files:
        with open(pkl_path, "rb") as f:
            obj = pickle.load(f, encoding="latin1")  # на всякий случай
        if is_annotation_dict(obj):
            annotation = obj
        else:
            transcription = obj

    assert annotation is not None, "Не удалось распознать файл с аннотациями"
    assert transcription is not None, "Не удалось распознать файл с транскрипциями"

    return annotation, transcription

train_annotation, train_transcription = load_annotation_and_transcription(TRAIN_DIR)
val_annotation, val_transcription = load_annotation_and_transcription(VAL_DIR)

print("Ключи верхнего уровня в annotation:", list(train_annotation.keys()))
some_trait = list(train_annotation.keys())[5]
print(f"Пример внутреннего словаря для признака '{some_trait}':")
items = list(train_annotation[some_trait].items())[:5]
for vid_name, value in items:
    print(vid_name, "->", value)

print("\nПримеры транскриптов:")
for vid_name, text in list(train_transcription.items())[:5]:
    print("Видео:", vid_name)
    print("Текст:", text[:120], "...")
    print("---")

Ключи верхнего уровня в annotation: ['extraversion', 'neuroticism', 'agreeableness', 'conscientiousness', 'interview', 'openness']
Пример внутреннего словаря для признака 'openness':
J4GQm9j0JZ0.003.mp4 -> 0.4888888888888889
zEyRyTnIw5I.005.mp4 -> 0.3666666666666667
nskJh7v6v1U.004.mp4 -> 0.5111111111111111
6wHQsN5g2RM.000.mp4 -> 0.3777777777777778
dQOeQYWIgm8.000.mp4 -> 0.6222222222222222

Примеры транскриптов:
Видео: J4GQm9j0JZ0.003.mp4
Текст: He's cutting it and then turn around and see the end result, but I'm glad he didn't do that because I probably would've  ...
---
Видео: zEyRyTnIw5I.005.mp4
Текст: Responsibility to house the organ I had been given and I needed to tell them I was going to take good care of that organ ...
---
Видео: nskJh7v6v1U.004.mp4
Текст: I actually got quite a few sets of black pens this year, because I bought one pack. I think I bought two packs, actually ...
---
Видео: 6wHQsN5g2RM.000.mp4
Текст: I ate a lot. I'd like a lot of foods. I remember I have favor

### Список признаков (Big Five + interview)

Нормально проверить, что среди ключей аннотаций есть:
- `extraversion`, `agreeableness`, `conscientiousness`, `neuroticism`, `openness`, `interview`.


## 3. Подготовка видео: выбор кадров и преобразования

Для VSSD нам нужны "картинки" 3×224×224:
- выбираем фиксированное число кадров (например, 16) равномерно по ролику;
- применяем те же аугментации/нормировку, что и при обучении на ImageNet (минимальный вариант).


## Добавила cv2.setNumThreads(0)
torch.backends.cudnn.benchmark = True

In [43]:
import cv2
import torch
import numpy as np
from pathlib import Path
from typing import Callable
from torchvision import transforms

cv2.setNumThreads(0)
torch.backends.cudnn.benchmark = True
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

NUM_FRAMES = 32  # можно увеличить до 32 на мощном кластере

IMG_SIZE = 224

# трансформации ожидают CHW-тензор float32 в [0, 1]
train_frame_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), antialias=True),  # умеет работать с Tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_frame_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), antialias=True),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


def sample_indices(num_frames_total: int, num_samples: int) -> np.ndarray:
    """Равномерно выбираем num_samples индексов из диапазона [0, num_frames_total-1]."""
    if num_frames_total <= num_samples:
        # если кадров мало, просто повторяем последние
        indices = np.linspace(0, num_frames_total - 1, num_frames_total, dtype=int)
        pad = np.full(num_samples - num_frames_total, num_frames_total - 1, dtype=int)
        return np.concatenate([indices, pad])
    else:
        return np.linspace(0, num_frames_total - 1, num_samples, dtype=int)


def load_video_frames(
    video_path: Path,
    num_frames: int,
    transform: Callable,
) -> torch.Tensor:
    """
    Читает видео и возвращает тензор формы [T, C, H, W].

    video_path: путь к .mp4
    num_frames: сколько кадров взять на видео
    transform: torchvision.transforms, ожидающий тензор [C, H, W] в [0, 1]
    """

    cap = cv2.VideoCapture(str(video_path))
    frames_bgr = []

    if not cap.isOpened():
        raise RuntimeError(f"Не удалось открыть видео: {video_path}")

    while True:
        ret, frame_bgr = cap.read()
        if not ret:
            break
        frames_bgr.append(frame_bgr)

    cap.release()

    if len(frames_bgr) == 0:
        raise RuntimeError(f"Видео {video_path} не содержит кадров")

    total = len(frames_bgr)

    # выбираем индексы кадров
    if total >= num_frames:
        indices = np.linspace(0, total - 1, num_frames, dtype=int)
    else:
        indices = list(range(total)) + [total - 1] * (num_frames - total)

    frames = []
    for idx in indices:
        frame_bgr = frames_bgr[idx]                             # H x W x 3 (BGR)
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)  # -> RGB

        # КРИТИЧЕСКИЙ МОМЕНТ: пересоздаём массив через наш numpy
        frame_rgb_np = np.asarray(frame_rgb, dtype=np.uint8).copy()

        # numpy [H, W, C] -> torch [C, H, W], float32 в [0, 1]
        frame_tensor = torch.from_numpy(frame_rgb_np).permute(2, 0, 1).float().div_(255.0)

        # transform должен работать с Tensor
        frame_tensor = transform(frame_tensor)                  # [C, H, W]
        frames.append(frame_tensor)

    frames = torch.stack(frames, dim=0)                         # [T, C, H, W]
    return frames


## 4. Маппинг `video_name` → таргеты Big Five

Аннотации хранятся в виде:

```python
annotation['extraversion']['video_0001'] = 0.73
```

Для обучения нам нужен единый вектор таргетов:

```python
y = [extraversion, agreeableness, conscientiousness, neuroticism, openness]
```


In [44]:
BIG_FIVE_ORDER = [
    "extraversion",
    "agreeableness",
    "conscientiousness",
    "neuroticism",
    "openness",
]

def build_targets_dict(annotation: Dict[str, Dict[str, float]]) -> Dict[str, np.ndarray]:
    """
    Строит словарь:
      video_name -> np.array shape [5] (Big Five в фиксированном порядке)
    """
    targets = {}
    # возьмём список всех видео из одного из признаков
    some_trait = BIG_FIVE_ORDER[0]
    video_names = list(annotation[some_trait].keys())
    for vid in video_names:
        vals = []
        for trait in BIG_FIVE_ORDER:
            vals.append(annotation[trait][vid])
        targets[vid] = np.array(vals, dtype=np.float32)
    return targets

train_targets = build_targets_dict(train_annotation)
val_targets = build_targets_dict(val_annotation)

# print("Пример таргета:")
# for vid, y in list(train_targets.items())[:5]:
#     print(vid, "->", y)

Пример таргета:
J4GQm9j0JZ0.003.mp4 -> [0.5233645  0.62637365 0.60194176 0.5520833  0.4888889 ]
zEyRyTnIw5I.005.mp4 -> [0.34579438 0.47252747 0.5825243  0.375      0.36666667]
nskJh7v6v1U.004.mp4 -> [0.25233644 0.4065934  0.4854369  0.29166666 0.51111114]
6wHQsN5g2RM.000.mp4 -> [0.45794392 0.50549453 0.39805827 0.48958334 0.37777779]
dQOeQYWIgm8.000.mp4 -> [0.60747665 0.4065934  0.6213592  0.48958334 0.62222224]


## 6. Скрипт извлечения признаков VSSD

Эту ячейку нужно прогнать **один раз** на кластере.

Что делает код:
- читает видео из `TRAIN_DIR` и `VAL_DIR` через `FirstImpressionsVideoDataset`;
- выбирает фиксированное число кадров, применяет аугментации / преобразования;
- прогоняет кадры через VSSD-бэкбон (`backbone`);
- усредняет признаки по времени и получает вектор размера `feat_dim` для каждого видео;
- сохраняет по одному `.pt` файлу на видео в `features_vssd/train` и `features_vssd/val`.

Дальше в ноутбуке обучение идёт уже **по этим сохранённым признакам**, поэтому к этой ячейке возвращаться не нужно (если только не меняется архитектура бэкбона или не появляется новая разметка).

In [ ]:
# ===== ОДНОРАЗОВАЯ ВЫГРУЗКА ПРИЗНАКОВ (VSSD) =====
from torch.utils.data import DataLoader

FEATURES_ROOT = Path("features_vssd")
TRAIN_FEATURES_DIR = FEATURES_ROOT / "train"
VAL_FEATURES_DIR   = FEATURES_ROOT / "val"

TRAIN_FEATURES_DIR.mkdir(parents=True, exist_ok=True)
VAL_FEATURES_DIR.mkdir(parents=True, exist_ok=True)


def extract_split_features(dataset, out_dir: Path, batch_size: int = 32):
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    model.eval()          # VSSDVideoRegressor
    with torch.no_grad():
        for batch in loader:
            frames = batch["frames"].to(device, non_blocking=True).float()
            ids = batch["video_name"]   # БЫЛО "video_id" — ТАК ПРАВИЛЬНО

            # прогоняем ТОЛЬКО через бэкбон
            B, T, C, H, W = frames.shape
            x = frames.view(B * T, C, H, W)
            with amp.autocast(enabled=False):  # внутри VSSD нужно float32
                feats = backbone(x)            # [B*T, D]

            # усредняем по времени, получаем [B, D]
            feats = feats.view(B, T, -1).mean(dim=1).cpu()

            for vid, f in zip(ids, feats):
                torch.save(f, out_dir / f"{vid}.pt")


# ЭТОТ БЛОК НУЖНО ВЫПОЛНИТЬ ОДИН РАЗ
extract_split_features(train_ds, TRAIN_FEATURES_DIR)
extract_split_features(val_ds,   VAL_FEATURES_DIR)

In [45]:
from typing import Dict, List, Tuple, Optional

class FirstImpressionsVideoDataset(Dataset):
    def __init__(
        self,
        phase_dir: Path,
        targets: Optional[Dict[str, np.ndarray]],
        num_frames: int = 16,
        train: bool = True,
    ):
        # базовые поля
        self.phase_dir = phase_dir
        self.targets = targets
        self.num_frames = num_frames
        self.train = train

        # собираем все mp4 в директории
        self.video_paths = sorted([p for p in phase_dir.glob("*.mp4")])
        assert self.video_paths, f"Не найдено mp4 в {phase_dir}"

        # список ID (как они называются в словаре targets)
        self.video_ids: List[str] = []

        self.frame_transform = train_frame_transform if train else eval_frame_transform

        if self.targets is not None:
            # фильтруем только те видео, у которых есть аннотация
            filtered_paths = []
            filtered_ids = []

            for p in self.video_paths:
                stem = p.stem          # без .mp4
                name = p.name          # с .mp4
                key = None

                # пробуем разные варианты названия
                if stem in self.targets:
                    key = stem
                elif name in self.targets:
                    key = name

                if key is not None:
                    filtered_paths.append(p)
                    filtered_ids.append(key)

            self.video_paths = filtered_paths
            self.video_ids = filtered_ids

            print(f"[{phase_dir.name}] всего mp4: {len(self.video_paths)}, "
                  f"с совпавшими аннотациями: {len(self.video_ids)}")

            # если после фильтрации не осталось видео — сразу падаем с понятным сообщением
            assert len(self.video_paths) > 0, (
                f"В {phase_dir} есть видео, но ни одно имя не совпало с аннотациями. "
                f"Проверь, как называются ключи в pkl и файлы .mp4"
            )
        else:
            # тестовый датасет без таргетов
            self.video_ids = [p.stem for p in self.video_paths]
            print(f"[{phase_dir.name}] видео (без таргетов): {len(self.video_paths)}")

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx: int):
        # на всякий случай проверяем индекс
        if idx < 0 or idx >= len(self.video_paths):
            raise IndexError(f"Index {idx} out of range for dataset of size {len(self.video_paths)}")

        video_path = self.video_paths[idx]
        vid_id = self.video_ids[idx]  # ключ для словаря targets

        frames = load_video_frames(
            video_path=video_path,
            num_frames=self.num_frames,
            transform=self.frame_transform,
        )  # [T, C, H, W]

        if self.targets is None:
            target = np.zeros(len(BIG_FIVE_ORDER), dtype=np.float32)
        else:
            target = self.targets[vid_id]

        return {
            "video_name": vid_id,
            "frames": frames,
            "target": torch.from_numpy(target),
        }

train_ds = FirstImpressionsVideoDataset(TRAIN_DIR, train_targets, num_frames=NUM_FRAMES, train=True)
val_ds   = FirstImpressionsVideoDataset(VAL_DIR,  val_targets,   num_frames=NUM_FRAMES, train=False)

print("len(train_ds) =", len(train_ds))
print("len(val_ds)   =", len(val_ds))

sample = train_ds[0]
print("video_name:", sample["video_name"])
print("frames shape:", sample["frames"].shape)  # должно быть [T, C, H, W]
print("target:", sample["target"])

[TRAIN] всего mp4: 6000, с совпавшими аннотациями: 6000
[VALIDATION] всего mp4: 2000, с совпавшими аннотациями: 2000
len(train_ds) = 6000
len(val_ds)   = 2000
video_name: --Ymqszjv54.001.mp4
frames shape: torch.Size([32, 3, 224, 224])
target: tensor([0.5514, 0.5275, 0.6505, 0.5000, 0.7444])


In [ ]:
# ===== ДАТАСЕТ ПО ГОТОВЫМ ПРИЗНАКАМ VSSD =====
class FirstImpressionsFeaturesDataset(Dataset):
    def __init__(self, features_dir: Path, targets: Dict[str, np.ndarray]):
        self.features_dir = Path(features_dir)
        self.targets = targets
        # используем те же video_name, что и в train_targets / val_targets
        self.video_ids = sorted(targets.keys())

    def __len__(self) -> int:
        return len(self.video_ids)

    def __getitem__(self, idx: int):
        vid_id = self.video_ids[idx]
        feat_path = self.features_dir / f"{vid_id}.pt"
        feat = torch.load(feat_path, map_location="cpu")  # [D], float32

        target = self.targets[vid_id]

        return {
            "video_name": vid_id,
            # Ключ оставляем "frames", чтобы train_one_epoch НЕ переписывать
            "frames": feat.float(),                # [D]
            "target": torch.from_numpy(target),    # [5]
        }


# Датасеты для обучения по признакам
train_feats_ds = FirstImpressionsFeaturesDataset(TRAIN_FEATURES_DIR, train_targets)
val_feats_ds   = FirstImpressionsFeaturesDataset(VAL_FEATURES_DIR,  val_targets)

print("len(train_feats_ds) =", len(train_feats_ds))
print("len(val_feats_ds)   =", len(val_feats_ds))
sample = train_feats_ds[0]
print("shape(features) =", sample["frames"].shape)


### DataLoader-ы для обучения по признакам

Здесь создаются `train_loader` и `val_loader` поверх `FirstImpressionsFeaturesDataset`.

- `BATCH_SIZE` — сколько векторов признаков (видео) идёт в один шаг обучения.
- `NUM_WORKERS` берётся из переменной окружения `SLURM_CPUS_PER_TASK`, что позволяет задействовать все выделенные CPU-потоки при чтении `.pt` файлов.
- `pin_memory=True` ускоряет перенос батчей на GPU.
- `persistent_workers=True` и `prefetch_factor` помогают не тратить время на постоянное создание/уничтожение воркеров и заранее подготавливать следующие батчи.


In [56]:
import os
from torch.utils.data import DataLoader

BATCH_SIZE = 64  # размер батча по признакам (увеличивай, пока помещается в память GPU)
NUM_WORKERS = int(os.getenv("SLURM_CPUS_PER_TASK", 8))
print(f"NUM_WORKERS = {NUM_WORKERS}, BATCH_SIZE = {BATCH_SIZE}")

train_loader = DataLoader(
    train_feats_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,      # подготавливаем несколько батчей заранее, чтобы не ждать диск
)

val_loader = DataLoader(
    val_feats_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,      # подготавливаем несколько батчей заранее, чтобы не ждать диск
)


## 6. Интеграция VSSD

Здесь важный момент.

В оригинальном репозитории VSSD:
- есть готовые конфиги и скрипты для **ImageNet / COCO / ADE20K**;:contentReference[oaicite:1]{index=1}
- бэкбон VSSD обучен на картинках формата 3×224×224.

Нам нужно:
- импортировать **предобученную** модель VSSD (Tiny/Small/Base);
- использовать её как **encoder** (feature extractor), без финального классификатора;
- поверх построить свою видео-голову.

Ниже – «обертка», которую нужно адаптировать под реальные классы в репозитории VSSD.

Вариант:
1. Клонируешь репозиторий `VSSD`:
   ```bash
   git clone https://github.com/YuHengsss/VSSD.git
   cd VSSD/classification
   pip install -r ../requirements.txt
   ```
2. В ноутбуке добавляешь путь `VSSD/classification` в `sys.path`.
3. Импортируешь нужный класс модели, например `VSSD_Tiny_iccv2025` (имя смотри в исходниках конфигов).


In [57]:
from types import SimpleNamespace

cfg_path = VSSD_CLASSIFICATION_ROOT / "configs" / "vssd" / "vssd_base_e300.yaml"  # подставь реальный .yaml
args = SimpleNamespace(
    cfg=str(cfg_path),
    opts=None,              # доп. правки конфига через список ключ–значение, можно оставить None
    batch_size=None,
    data_path=None,
    zip=None,
    cache_mode=None,
    pretrained=None,
    resume=None,
    accumulation_steps=None,
    use_checkpoint=None,
    disable_amp=None,
    output="./output",
    tag="debug",
    eval=False,
    throughput=False,
    traincost=False,
    enable_persistance=False,
    enable_amp=None,
    fused_layernorm=None,
    optim=None,
    ddp="torch",
)

config = get_config(args)
model = build_model(config)
model.eval()


=> merge config from VSSD/classification/configs/vssd/vssd_base_e300.yaml


VMAMBA2(
  (patch_embed): Stem(
    (conv1): ConvLayer(
      (conv): Conv2d(3, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (norm): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): ReLU()
    )
    (conv2): Sequential(
      (0): ConvLayer(
        (conv): Conv2d(48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (norm): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act): ReLU()
      )
      (1): ConvLayer(
        (conv): Conv2d(48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (norm): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (conv3): Sequential(
      (0): ConvLayer(
        (conv): Conv2d(48, 384, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (norm): BatchNorm2d(384, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
 

### Обертка VSSDBackbone

Предполагаем, что:
- вызов `backbone(x)` возвращает признаки размерности `[B, D]` (или logits);
- у модели есть атрибут `num_features` или похожий для размерности выхода.

Если VSSD возвращает logits, нужно отбросить финальный `head` и использовать предпоследний слой.
Это уже зависит от конкретной реализации – здесь оставляю пометки.


In [58]:
import torch
import torch.nn as nn
from torch.cuda import amp
torch.backends.cudnn.benchmark = True

from config import _C
from models import build_model


class VSSDBackbone(nn.Module):
    """
    Обёртка над VMAMBA2 из VSSD/classification:
    - правильно настраивает конфиг
    - строит модель через build_model(cfg)
    - хранит размер выходного признакового вектора feat_dim
    """
    def __init__(
        self,
        img_size: int = 224,
        pretrained_ckpt: Optional[str] = None,  # на будущее, сейчас можно оставить None
        device: torch.device = torch.device("cuda")
    ):
        super().__init__()
        self.device = device

        # 1) Берём базовый конфиг и настраиваем его под VMAMBA2
        cfg = _C.clone()
        cfg.defrost()

        # размер входного изображения
        cfg.DATA.IMG_SIZE = img_size

        # КРИТИЧНО: тип модели должен быть vmamba2, иначе build_model вернёт None
        cfg.MODEL.TYPE = "vmamba2"

        # (опционально) имя модели
        cfg.MODEL.NAME = "vmamba2_tiny_224"

        # (по желанию, но обычно дефолты уже корректные)
        # cfg.MODEL.VMAMBA2.PATCH_SIZE = 4
        # cfg.MODEL.VMAMBA2.EMBED_DIM = 96
        # ...

        cfg.freeze()

        # 2) Строим модель
        backbone = build_model(cfg)  # is_pretrain=False по умолчанию
        assert backbone is not None, (
            "build_model(cfg) вернул None. "
            "Проверь, что cfg.MODEL.TYPE == 'vmamba2'."
        )

        # 3) Переносим на нужное устройство и ставим eval()
        backbone.to(self.device)
        backbone.eval()

        self.backbone = backbone

        # 4) Определяем размер признаков
        if hasattr(backbone, "num_features"):
            feat_dim = backbone.num_features
        elif hasattr(backbone, "head") and hasattr(backbone.head, "in_features"):
            feat_dim = backbone.head.in_features
        else:
            # fallback, если в будущем изменят API
            raise RuntimeError(
                "Не удалось определить размер признаков у VSSD-модели. "
                "Проверь атрибуты модели (num_features или head.in_features)."
            )

        self.feat_dim = feat_dim

    @torch.no_grad()
    def _forward_features(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: [B, 3, H, W] на ТОМ ЖЕ устройстве, что и self.backbone (обычно cuda)
        return: [B, D]
        """
        # Важно: НЕ переводим x на CPU, иначе Triton (mamba_ssm) падает с "cpu tensor?"
        # Просто отключаем autocast, потому что внутри VMAMBA2 всё ожидает float32
        with amp.autocast(enabled=False):
            feats = self.backbone.forward_features(x)  # [B, D] для VMAMBA2

        return feats

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Обёртка для прямого вызова: x -> признаки
        """
        return self._forward_features(x)


class VSSDVideoRegressor(nn.Module):
    """
    Модель для регрессии big-five по видео:
    - backbone: VSSDBackbone (кадровый)
    - голова: MLP по усреднённым признакам по времени
    """
    def __init__(self, backbone: VSSDBackbone, num_traits: int):
        super().__init__()
        self.backbone = backbone
        self.feat_dim = backbone.feat_dim
        self.num_traits = num_traits

        # Простая голова: LayerNorm + Linear
        self.head = nn.Sequential(
            nn.LayerNorm(self.feat_dim),
            nn.Linear(self.feat_dim, self.num_traits),
        )

    def forward(self, frames: torch.Tensor) -> torch.Tensor:
        """
        frames: [B, T, C, H, W]
        return: [B, num_traits]
        """
        B, T, C, H, W = frames.shape

        # Склеиваем время в батч и прогоняем через VSSD
        x = frames.view(B * T, C, H, W)     # [B*T, C, H, W]
        feats = self.backbone(x)           # [B*T, D]

        # Возвращаем разбиение по времени
        feats = feats.view(B, T, -1)       # [B, T, D]

        # Усредняем по времени
        feats_mean = feats.mean(dim=1)     # [B, D]

        # Прогоняем через голову
        out = self.head(feats_mean)        # [B, num_traits]
        return out

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

IMG_SIZE = 224
PRETRAINED_CKPT = None  # если потом будет чекпоинт VSSD — сюда путь

# Бэкбон VSSD (VMAMBA2)
backbone = VSSDBackbone(
    img_size=IMG_SIZE,
    pretrained_ckpt=PRETRAINED_CKPT,
    device=device,
)

# Видеорегрессор поверх бэкбона
model = VSSDVideoRegressor(
    backbone=backbone,
    num_traits=len(BIG_FIVE_ORDER),   # у тебя уже есть BIG_FIVE_ORDER
)

model.to(device)

print("feat_dim:", backbone.feat_dim)
print("Модель готова к обучению")


Using device: cuda
feat_dim: 512
Модель готова к обучению


### Модель-регрессор по признакам

На этом этапе VSSD уже посчитала визуальные признаки для каждого видео.
Модель-регрессор получает на вход вектор признаков фиксированного размера `feat_dim` и предсказывает 5 непрерывных значений (Big Five) в диапазоне [0, 1].

Архитектура головы:
- `LayerNorm(feat_dim)` — нормализация признаков;
- `Linear(feat_dim → 5)` — линейный слой, выдающий значения для пяти черт.

Такую голову легко доработать (например, добавить ещё один скрытый слой), не трогая VSSD-бэкбон и пайплайн извлечения признаков.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# определим размер признаков по одному примеру
feat_dim = train_feats_ds[0]["frames"].shape[-1]
print("feat_dim (from saved features):", feat_dim)


class FeaturesRegressor(nn.Module):
    def __init__(self, feat_dim: int, num_traits: int):
        super().__init__()
        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Linear(feat_dim, num_traits),
        )

    def forward(self, feats: torch.Tensor) -> torch.Tensor:
        # feats: [B, D]
        return self.head(feats)


regressor = FeaturesRegressor(
    feat_dim=feat_dim,
    num_traits=len(BIG_FIVE_ORDER),
).to(device)

print(regressor)


## 8. Целевая функция и метрики

Для Big Five обычно используют:
- MSE / MAE;
- корреляцию Пирсона / R².

Здесь сделаем:
- loss = MSE;
- метрики: MAE и корреляция Пирсона по каждой из 5 шкал.

In [59]:
import numpy as np
from typing import Dict, Tuple
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler

# Порядок черт – как в твоём коде / данных
TRAIT_NAMES = BIG_FIVE_ORDER  # ['extraversion', 'neuroticism', ...]


def _pearsonr_np(x: np.ndarray, y: np.ndarray) -> float:
    """Коэффициент корреляции Пирсона, без scipy, с защитой от нулевой дисперсии."""
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)

    x_mean = x.mean()
    y_mean = y.mean()

    x_centered = x - x_mean
    y_centered = y - y_mean

    num = np.sum(x_centered * y_centered)
    den = np.sqrt(np.sum(x_centered ** 2) * np.sum(y_centered ** 2)) + 1e-8

    if den == 0.0:
        return 0.0
    return float(num / den)


def compute_metrics(pred: torch.Tensor, target: torch.Tensor) -> Dict[str, float]:
    """
    Базовые метрики для логов обучения:
    - MAE по каждой черте (MAE_trait)
    - Corr по каждой черте (Corr_trait)
    - MAE_mean
    - Corr_mean

    pred, target: [N, 5] в [0, 1]
    """
    preds = pred.detach().cpu().numpy()    # [N, 5]
    targets = target.detach().cpu().numpy()

    # MAE по чертам
    mae_per = np.mean(np.abs(preds - targets), axis=0)   # [5]

    # Корреляция Пирсона по чертам
    corr_per = []
    for i in range(preds.shape[1]):
        r = _pearsonr_np(preds[:, i], targets[:, i])
        corr_per.append(r)

    metrics: Dict[str, float] = {}

    # Разворачиваем поимённо по BIG_FIVE_ORDER
    for i, trait in enumerate(TRAIT_NAMES):
        metrics[f"MAE_{trait}"] = float(mae_per[i])
        metrics[f"Corr_{trait}"] = float(corr_per[i])

    # Средние по всем чертам
    metrics["MAE_mean"] = float(np.mean(mae_per))
    metrics["Corr_mean"] = float(np.mean(corr_per))

    return metrics


def compute_regression_metrics(
    all_preds: torch.Tensor,
    all_targets: torch.Tensor,
) -> Dict[str, object]:
    """
    Дополнительные регрессионные метрики:
    - MAE_per: {trait -> mae}
    - Corr_per: {trait -> corr}
    - R2_mean
    - R2_per: {trait -> r2}

    all_preds, all_targets: torch.Tensor [N, 5] на CPU или GPU
    """
    preds = all_preds.detach().cpu().numpy()   # [N, 5]
    targets = all_targets.detach().cpu().numpy()

    # MAE по чертам
    mae_per = np.mean(np.abs(preds - targets), axis=0)  # [5]

    # Pearson r по чертам
    corr_per = []
    for i in range(preds.shape[1]):
        r = _pearsonr_np(preds[:, i], targets[:, i])
        corr_per.append(r)

    # R^2 по чертам
    r2_per = []
    for i in range(preds.shape[1]):
        y = targets[:, i]
        y_hat = preds[:, i]
        ss_res = np.sum((y - y_hat) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)

        if ss_tot < 1e-8:
            r2 = 0.0
        else:
            r2 = 1.0 - ss_res / (ss_tot + 1e-8)

        r2_per.append(float(r2))

    r2_mean = float(np.mean(r2_per))

    metrics = {
        "MAE_per": {trait: float(v) for trait, v in zip(TRAIT_NAMES, mae_per)},
        "Corr_per": {trait: float(v) for trait, v in zip(TRAIT_NAMES, corr_per)},
        "R2_mean": r2_mean,
        "R2_per": {trait: float(v) for trait, v in zip(TRAIT_NAMES, r2_per)},
    }
    return metrics

### Циклы train / validate

Используем:
- `torch.cuda.amp.autocast` для mixed precision;
- `GradScaler` для стабильного обучения.

In [60]:
import time
from datetime import datetime
import math

def format_time(seconds: float) -> str:
    seconds = int(seconds)
    m, s = divmod(seconds, 60)
    h, m = divmod(m, 60)
    if h > 0:
        return f"{h:02d}:{m:02d}:{s:02d}"
    else:
        return f"{m:02d}:{s:02d}"


In [61]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scaler: GradScaler,
    device: torch.device,
) -> Tuple[float, Dict[str, float]]:
    model.train()
    total_loss = 0.0
    all_pred = []
    all_targ = []

    for batch in loader:
        frames = batch["frames"].to(device, non_blocking=True).float()
        target = batch["target"].to(device, non_blocking=True).float()

        optimizer.zero_grad(set_to_none=True)

        # pred = model(frames)            # [B, 5]
        # loss = nn.MSELoss()(pred, target)
        with autocast():
            pred = model(frames)
            loss = nn.MSELoss()(pred, target)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * frames.size(0)
        all_pred.append(pred.detach())
        all_targ.append(target.detach())

    all_pred = torch.cat(all_pred, dim=0)   # [N, 5]
    all_targ = torch.cat(all_targ, dim=0)   # [N, 5]

    epoch_loss = total_loss / len(loader.dataset)
    metrics = compute_metrics(all_pred, all_targ)
    return epoch_loss, metrics

def validate(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
) -> Tuple[float, Dict[str, object]]:
    model.eval()
    total_loss = 0.0
    all_pred = []
    all_targ = []

    with torch.no_grad():
        for batch in loader:
            frames = batch["frames"].to(device, non_blocking=True).float()
            target = batch["target"].to(device, non_blocking=True).float()

            pred = model(frames)
            loss = nn.MSELoss()(pred, target)

            total_loss += loss.item() * frames.size(0)
            all_pred.append(pred.detach())
            all_targ.append(target.detach())

    all_pred = torch.cat(all_pred, dim=0)   # [N, 5]
    all_targ = torch.cat(all_targ, dim=0)   # [N, 5]

    epoch_loss = total_loss / len(loader.dataset)

    # Базовые метрики (MAE_mean, Corr_mean, MAE_trait, Corr_trait)
    base_metrics = compute_metrics(all_pred, all_targ)
    # Дополнительные регрессионные (MAE_per, Corr_per, R2_*)
    extra_metrics = compute_regression_metrics(all_pred, all_targ)

    # Объединяем; ключи MAE_mean / Corr_mean берём из base_metrics
    metrics: Dict[str, object] = {**base_metrics, **extra_metrics}
    return epoch_loss, metrics

## 9. Запуск обучения

Для начала – прототип на одной GPU (или CPU, если нужно только проверить код).

На кластере:
- увеличиваешь `EPOCHS`, `BATCH_SIZE`, `NUM_FRAMES`;
- переходишь на запуск через `torchrun` и DistributedDataParallel.

In [62]:
# === ГИПЕРПАРАМЕТРЫ ОБУЧЕНИЯ ===

LEARNING_RATE = 1e-4
WEIGHT_DECAY  = 1e-4

# "умное" число эпох: максимум и ранняя остановка
MAX_EPOCHS        = 20     # верхняя граница, можно поднять/опустить
ES_PATIENCE       = 5      # сколько эпох ждать без улучшения val_loss
ES_MIN_DELTA      = 1e-4   # насколько должен улучшиться val_loss

# === ОПТИМИЗАТОР И СКЕЖУЛЕР ===

optimizer = torch.optim.AdamW(
    regressor.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scaler = GradScaler(enabled=torch.cuda.is_available())

# уменьшаем lr, если val_loss не улучшается
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
    verbose=True,
)

# === ИСТОРИЯ И ПЕРЕМЕННЫЕ ДЛЯ EARLY STOPPING ===

history = {
    "train_loss": [],
    "val_loss": [],
    "val_corr_mean": [],
    "val_mae_mean": [],
}

best_val_loss = math.inf
best_epoch = 0
epochs_no_improve = 0

ckpt_dir = Path("checkpoints")
ckpt_dir.mkdir(exist_ok=True, parents=True)

# === ЗАПУСК ОБУЧЕНИЯ ===

training_start_time = time.time()
print(f"Старт обучения: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

for epoch in range(1, MAX_EPOCHS + 1):
    epoch_start_time = time.time()

    progress_before = (epoch - 1) / MAX_EPOCHS * 100
    print("\n" + "=" * 70)
    print(
        f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] "
        f"Эпоха {epoch}/{MAX_EPOCHS} "
        f"(начало, прогресс: {progress_before:.1f}%)"
    )

    # --- ОБУЧЕНИЕ НА TRAIN ---
    train_loss, train_metrics = train_one_epoch(
        regressor, train_loader, optimizer, scaler, device
    )

    val_loss, val_metrics = validate(
        regressor, val_loader, device
    )
    # --- ОБНОВЛЯЕМ СКЕЖУЛЕР ---
    scheduler.step(val_loss)

    # --- ВРЕМЯ ---
    epoch_time = time.time() - epoch_start_time
    total_time = time.time() - training_start_time
    progress_after = epoch / MAX_EPOCHS * 100

    # --- ЛОГИ ПО ЭПОХЕ ---
    print(
        f"Эпоха {epoch} завершена "
        f"(прогресс: {progress_after:.1f}%)"
    )
    print(
        f"  Время на эпоху: {format_time(epoch_time)}, "
        f"общее время обучения: {format_time(total_time)}"
    )

    print(
        f"  Train: loss={train_loss:.4f}, "
        f"MAE_mean={train_metrics['MAE_mean']:.4f}, "
        f"Corr_mean={train_metrics['Corr_mean']:.4f}"
    )
    print(
        f"  Val  : loss={val_loss:.4f}, "
        f"MAE_mean={val_metrics['MAE_mean']:.4f}, "
        f"Corr_mean={val_metrics['Corr_mean']:.4f}"
    )

    # --- СОХРАНЯЕМ ИСТОРИЮ ---
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_corr_mean"].append(val_metrics["Corr_mean"])
    history["val_mae_mean"].append(val_metrics["MAE_mean"])

    # --- СОХРАНЯЕМ ЧЕКПОИНТ ЭПОХИ ---
    ckpt_path = ckpt_dir / f"vssd_first_impressions_epoch{epoch:03d}.pth"
    torch.save(
        {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "history": history,
        },
        ckpt_path,
    )

    # --- EARLY STOPPING ПО val_loss ---
    if val_loss + ES_MIN_DELTA < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        epochs_no_improve = 0

        best_ckpt_path = ckpt_dir / "vssd_first_impressions_best.pth"
        torch.save(
            {
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "history": history,
            },
            best_ckpt_path,
        )
        print(
            f"Новый лучший val_loss: {best_val_loss:.6f} "
            f"(эпоха {epoch}), чекпоинт сохранён в {best_ckpt_path}"
        )
    else:
        epochs_no_improve += 1
        print(
            f"  val_loss не улучшился {epochs_no_improve} раз(а) подряд "
            f"(лучшая эпоха: {best_epoch}, best_val_loss={best_val_loss:.6f})"
        )

    if epochs_no_improve >= ES_PATIENCE:
        print(
            f"\nEARLY STOPPING: нет улучшений val_loss "
            f"{ES_PATIENCE} эпох подряд. Останавливаем обучение."
        )
        break

print("\nОбучение завершено.")
print(f"Лучшая эпоха: {best_epoch}, лучший val_loss: {best_val_loss:.6f}")
print(f"Общее время обучения: {format_time(time.time() - training_start_time)}")
print(f"Чекпоинты лежат в: {ckpt_dir}")


Старт обучения: 2025-12-08 21:49:12

[2025-12-08 21:49:12] Эпоха 1/20 (начало, прогресс: 0.0%)
Эпоха 1 завершена (прогресс: 5.0%)
  Время на эпоху: 37:58, общее время обучения: 37:58
  Train: loss=0.1056, MAE_mean=0.2284, Corr_mean=-0.0014
  Val  : loss=0.0390, MAE_mean=0.1573, Corr_mean=0.0149
Новый лучший val_loss: 0.039027 (эпоха 1), чекпоинт сохранён в checkpoints/vssd_first_impressions_best.pth

[2025-12-08 22:27:11] Эпоха 2/20 (начало, прогресс: 5.0%)


KeyboardInterrupt: 

## 10. Визуализация: кривые обучения и распределение таргетов

1. train/val loss по эпохам
2. средняя корреляция на валидации
3. гистограммы реальных значений Big Five


In [ ]:
# %%
# 1) Кривые loss
plt.figure(figsize=(6, 4))
plt.plot(history["train_loss"], label="train_loss")
plt.plot(history["val_loss"], label="val_loss")
plt.xlabel("Эпоха")
plt.ylabel("Loss (MSE)")
plt.title("Кривые обучения (MSE)")
plt.legend()
plt.grid(True)
plt.show()

# 2) Средняя корреляция
plt.figure(figsize=(6, 4))
plt.plot(history["val_corr_mean"], label="val_corr_mean")
plt.xlabel("Эпоха")
plt.ylabel("Средняя корреляция Пирсона")
plt.title("Качество на валидации (Corr_mean)")
plt.legend()
plt.grid(True)
plt.show()

# 3) Распределение таргетов по Big Five (train+val)
all_y = np.concatenate(
    [np.stack(list(train_targets.values()), axis=0),
     np.stack(list(val_targets.values()), axis=0)],
    axis=0
)

plt.figure(figsize=(12, 4))
for i, trait in enumerate(BIG_FIVE_ORDER):
    plt.subplot(1, len(BIG_FIVE_ORDER), i+1)
    plt.hist(all_y[:, i], bins=20, alpha=0.8)
    plt.title(trait)
    plt.xlabel("значение")
plt.tight_layout()
plt.show()


## 11. Scatter-плоты: предсказания vs ground truth

Для лучшего понимания, где модель ошибается, можно визуализировать:
- scatter `y_true` vs `y_pred` для каждого из пяти признаков.

In [ ]:
# %%
# Соберём предсказания на валидации
model.eval()
all_pred = []
all_targ = []

with torch.no_grad():
    for batch in val_loader:
        frames = batch["frames"].to(device, non_blocking=True)
        target = batch["target"].to(device, non_blocking=True)
        with autocast():
            pred = regressor(frames)
        all_pred.append(pred.cpu())
        all_targ.append(target.cpu())

all_pred = torch.cat(all_pred, dim=0).numpy()
all_targ = torch.cat(all_targ, dim=0).numpy()

plt.figure(figsize=(15, 3))
for i, trait in enumerate(BIG_FIVE_ORDER):
    plt.subplot(1, len(BIG_FIVE_ORDER), i+1)
    plt.scatter(all_targ[:, i], all_pred[:, i], alpha=0.3, s=8)
    plt.plot([0, 1], [0, 1], 'r--')
    plt.xlabel("GT")
    plt.ylabel("Pred")
    plt.title(trait)
    plt.xlim(0, 1)
    plt.ylim(0, 1)
plt.tight_layout()
plt.show()
